In [14]:
import os
import shutil
from datetime import datetime

import uproot
import pandas as pd
import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.colors as pcolors

file_path = "../clusters_seeds_island_79507-0.root_ntuplizer.root"
LABELS_CSV = "labeled_hits.csv"
ORIGINAL_BRANCHES = ["event", "hitID", "layer", "phi", "tbin", "zelem", "x", "y", "z", "adc"]
KEY_COLS = ["event", "hitID", "layer", "tbin"]   # row is per-tbin, so hitID alone is NOT unique
INT_COLS = ["event", "hitID", "layer", "tbin", "zelem"]
LABEL_COLORS = {"signal": "red", "noise": "blue", "unknown": "gray"}
NEXT_LABEL = {"unknown": "signal", "signal": "noise", "noise": "unknown"}
HITID_PALETTE = pcolors.qualitative.Dark24 + pcolors.qualitative.Light24

print("Loading real data for interactive labeling...")
with uproot.open(file_path) as f:
    hit_tree = f["ntp_hit"]
    df_real = hit_tree.arrays(ORIGINAL_BRANCHES, library="pd")
    print(f"Initial hits loaded: {len(df_real)}")

    # 1. Isolate the TPC layers
    df_real = df_real[(df_real['layer'] >= 7) & (df_real['layer'] <= 54)]
    # 2. Keep only signal/truth-associated hits (no background noise)
    df_real = df_real[df_real['hitID'] > 0]
    # 3. Ensure no zero-energy ghost hits
    df_real = df_real[df_real['adc'] > 0]
    # 4. Filter NaNs
    df_real = df_real.dropna(subset=['x', 'y', 'z', 'phi', 'tbin'])
    # 5. Enforce zelem bounds for TPC
    df_real = df_real[df_real['zelem'].isin([0, 1])]

    print(f"Valid TPC hits remaining after cuts: {len(df_real)}")

# Clean, contiguous index so it can be used as a stable key into customdata.
df_real = df_real.reset_index(drop=True)
df_real['label'] = 'unknown'
df_real['color'] = 'gray'

# NOTE: x = r*cos(phi), y = r*sin(phi) with r fixed per TPC layer (verified in
# sanity_check.ipynb), so a separate "transformed" (r*phi) view would be identical
# to the xyz view. We therefore keep just two views: raw Cartesian (xyz) and an
# "unrolled" (layer, phi, z) view that straightens helices into near-lines.


def _row_keys(frame):
    """Tuple key per row (plain ints) so dict lookups hash-match across DataFrames."""
    arr = frame[KEY_COLS].astype('int64').to_numpy()
    return [tuple(int(v) for v in row) for row in arr]


# ---- Resume: merge any previously saved labels back in -------------------------
if os.path.exists(LABELS_CSV):
    prev = pd.read_csv(LABELS_CSV)
    label_map = dict(zip(_row_keys(prev), prev['label']))
    restored = pd.Series(_row_keys(df_real), index=df_real.index).map(label_map)
    mask = restored.notna()
    df_real.loc[mask, 'label'] = restored[mask].values
    df_real['color'] = df_real['label'].map(LABEL_COLORS)
    print(f"Resumed {int(mask.sum())} previously-labeled hits from '{LABELS_CSV}'")

print(f"Successfully loaded {len(df_real)} hits. Ready for labeling!")


class LabelingUI:
    """Interactive signal/noise labeller.

    Batch labelling (fast):
      * Lasso a region in the 2D panel, then click Mark Signal / Mark Noise / Clear.
        In the 'unrolled' view over a layer range a whole track is a near-line, so one
        lasso grabs all of its hits at once.
      * Tick 'Click selects whole track' and click any point (2D *or* 3D): every hit
        sharing that hitID is selected, then one button labels the whole track. This is
        the practical way to batch from the 3D view (Plotly 3D has no lasso).

    Single point (fine fixes):
      * With 'Click selects whole track' off, a click cycles unknown->signal->noise and
        recolors every point stacked at the same plotted coordinate.

    Other:
      * Layer range / 'Whole Event' so a track reads as a continuous arc.
      * 'Color by hitID' overlays a categorical palette so candidate tracks pop out.
      * 'Save' writes labeled_hits.csv (timestamped backup first); reloading the kernel
        restores your labels via the resume step above.
    """

    MODES = ("xyz", "layer_phi_z")

    def __init__(self, mode="xyz"):
        assert mode in self.MODES, f"mode must be one of {self.MODES}"
        self.mode = mode
        self.last_selection = []      # df indices buffered from the last lasso / track-click
        self._coord_key = {}          # df_idx -> rounded 3D coordinate (overlap groups)

        small = mode == "layer_phi_z"
        self.fig3d = go.FigureWidget()
        self.fig3d.add_trace(go.Scatter3d(mode='markers', marker=dict(size=2 if small else 3), hoverinfo='text'))
        self.fig2d = go.FigureWidget()
        self.fig2d.add_trace(go.Scatter(mode='markers', marker=dict(size=5 if small else 4), hoverinfo='text'))

        if mode == "xyz":
            self.fig3d.update_layout(title="3D View (X, Y, Z)", margin=dict(l=0, r=0, b=0, t=40),
                                     scene=dict(aspectmode='data'))
            self.fig2d.update_layout(title="2D Slice: X vs Y", dragmode='lasso',
                                     margin=dict(l=0, r=0, b=0, t=40), xaxis_title="X", yaxis_title="Y")
        else:  # layer_phi_z  (unrolled)
            self.fig3d.update_layout(title="3D View (X=Layer, Y=Phi, Z=Z)", margin=dict(l=0, r=0, b=0, t=40),
                                     scene=dict(xaxis_title='Layer', yaxis_title='Phi', zaxis_title='Z',
                                                aspectmode='data'))
            self.fig2d.update_layout(title="2D Slice: Phi vs Z", dragmode='lasso',
                                     margin=dict(l=0, r=0, b=0, t=40), xaxis_title="Phi", yaxis_title="Z")

        self.trace3d = self.fig3d.data[0]
        self.trace2d = self.fig2d.data[0]

        events = sorted(df_real['event'].astype(int).unique())
        self.event_dropdown = widgets.Dropdown(options=events, value=events[0], description='Event:')

        self._lmin, self._lmax = int(df_real['layer'].min()), int(df_real['layer'].max())
        self.layer_range = widgets.IntRangeSlider(
            value=[self._lmin, self._lmin], min=self._lmin, max=self._lmax, step=1,
            description='Layers:', continuous_update=False, layout=widgets.Layout(width='420px'))
        self.whole_event_btn = widgets.Button(description='Whole Event', tooltip='Show every layer in this event')
        self.color_by_hitid = widgets.Checkbox(value=False, description='Color by hitID', indent=False)
        self.select_by_hitid = widgets.Checkbox(value=False, description='Click selects whole track', indent=False)

        self.btn_signal = widgets.Button(description='Mark Signal', button_style='danger')
        self.btn_noise = widgets.Button(description='Mark Noise', button_style='info')
        self.btn_clear = widgets.Button(description='Clear', button_style='warning')
        self.btn_save = widgets.Button(description='💾 Save', button_style='success')
        self.status = widgets.HTML()

        self.event_dropdown.observe(self.update_plots, 'value')
        self.layer_range.observe(self.update_plots, 'value')
        self.color_by_hitid.observe(self.update_plots, 'value')
        self.whole_event_btn.on_click(lambda _b: setattr(self.layer_range, 'value', (self._lmin, self._lmax)))
        self.btn_signal.on_click(lambda _b: self.apply_to_selection('signal'))
        self.btn_noise.on_click(lambda _b: self.apply_to_selection('noise'))
        self.btn_clear.on_click(lambda _b: self.apply_to_selection('unknown'))
        self.btn_save.on_click(lambda _b: self.save())

        self.trace2d.on_selection(self.on_selection)
        self.trace2d.on_click(self.on_click)
        self.trace3d.on_click(self.on_click)

        self.update_plots()

        self.ui = widgets.VBox([
            widgets.HBox([self.event_dropdown, self.layer_range, self.whole_event_btn]),
            widgets.HBox([self.color_by_hitid, self.select_by_hitid]),
            widgets.HBox([self.btn_signal, self.btn_noise, self.btn_clear, self.btn_save]),
            self.status,
            widgets.HBox([self.fig2d, self.fig3d]),
        ])

    # ----- data helpers ---------------------------------------------------------
    def _current_slice(self):
        lo, hi = self.layer_range.value
        return df_real[(df_real['event'] == self.event_dropdown.value)
                       & (df_real['layer'] >= lo) & (df_real['layer'] <= hi)]

    def _xyz(self, df_slice):
        """(x3, y3, z3) for the 3D figure and (x2, y2) for the 2D figure."""
        if self.mode == "xyz":
            return df_slice['x'], df_slice['y'], df_slice['z'], df_slice['x'], df_slice['y']
        # layer_phi_z (unrolled): straightens helices, z is the physical longitudinal axis
        return df_slice['layer'], df_slice['phi'], df_slice['z'], df_slice['phi'], df_slice['z']

    def _compute_colors(self, df_slice):
        if self.color_by_hitid.value:
            uniq = df_slice['hitID'].unique()
            cmap = {h: HITID_PALETTE[i % len(HITID_PALETTE)] for i, h in enumerate(uniq)}
            return df_slice['hitID'].map(cmap).tolist()
        return df_slice['color'].tolist()

    def update_plots(self, *args):
        df_slice = self._current_slice()
        x3, y3, z3, x2, y2 = self._xyz(df_slice)

        hover = [f"Index: {i}<br>hitID: {int(h)}<br>ADC: {a:.1f}<br>"
                 f"Layer: {int(l)} &nbsp; Tbin: {int(t)}<br>Label: {lb}"
                 for i, h, a, l, t, lb in zip(df_slice.index, df_slice['hitID'], df_slice['adc'],
                                              df_slice['layer'], df_slice['tbin'], df_slice['label'])]
        colors = self._compute_colors(df_slice)

        # Overlap groups: every df row plotted at the same rounded 3D coordinate.
        keys = list(zip(np.round(np.asarray(x3, float), 4),
                        np.round(np.asarray(y3, float), 4),
                        np.round(np.asarray(z3, float), 4)))
        self._coord_key = dict(zip(df_slice.index, keys))

        with self.fig3d.batch_update():
            self.trace3d.x, self.trace3d.y, self.trace3d.z = x3, y3, z3
            self.trace3d.marker.color = colors
            self.trace3d.customdata = df_slice.index
            self.trace3d.text = hover
        with self.fig2d.batch_update():
            self.trace2d.x, self.trace2d.y = x2, y2
            self.trace2d.marker.color = colors
            self.trace2d.customdata = df_slice.index
            self.trace2d.text = hover

        self._update_status(df_slice)

    def _update_status(self, df_slice):
        s = int((df_real['label'] == 'signal').sum())
        n = int((df_real['label'] == 'noise').sum())
        u = int((df_real['label'] == 'unknown').sum())
        ss = int((df_slice['label'] == 'signal').sum())
        sn = int((df_slice['label'] == 'noise').sum())
        self.status.value = (
            f"<b>Selection buffer:</b> {len(self.last_selection)} pts &nbsp;|&nbsp; "
            f"<b>This view:</b> {len(df_slice)} pts (signal {ss}, noise {sn}) &nbsp;|&nbsp; "
            f"<b>All events:</b> <span style='color:red'>signal {s}</span>, "
            f"<span style='color:blue'>noise {n}</span>, unknown {u}")

    # ----- labeling -------------------------------------------------------------
    def _set_labels(self, df_indices, label):
        df_real.loc[df_indices, 'label'] = label
        df_real.loc[df_indices, 'color'] = LABEL_COLORS[label]

    def _expand_overlap(self, df_idx):
        """All points sharing the clicked point's plotted 3D coordinate."""
        target = self._coord_key.get(df_idx)
        if target is None:
            return [df_idx]
        return [i for i, c in self._coord_key.items() if c == target]

    def apply_to_selection(self, label):
        if not self.last_selection:
            self.status.value = "<i>Lasso points (or click a track) first, then press a Mark/Clear button.</i>"
            return
        self._set_labels(self.last_selection, label)
        self.last_selection = []
        self.update_plots()

    def on_selection(self, trace, points, selector):
        self.last_selection = [int(trace.customdata[i]) for i in points.point_inds] if points.point_inds else []
        self._update_status(self._current_slice())

    def on_click(self, trace, points, selector):
        if not points.point_inds:
            return
        df_idx = int(trace.customdata[points.point_inds[0]])

        if self.select_by_hitid.value:
            # Batch-select the whole track segment visible in the current slice.
            hid = df_real.loc[df_idx, 'hitID']
            sl = self._current_slice()
            self.last_selection = sl.index[sl['hitID'] == hid].tolist()
            self.status.value = (f"<i>Selected {len(self.last_selection)} hits of hitID {int(hid)} "
                                 f"— press Mark Signal / Mark Noise / Clear. "
                                 f"(Use 'Whole Event' first to grab the full track.)</i>")
            return

        nxt = NEXT_LABEL[df_real.loc[df_idx, 'label']]
        self._set_labels(self._expand_overlap(df_idx), nxt)
        self.update_plots()

    # ----- persistence ----------------------------------------------------------
    def save(self):
        labeled = df_real[df_real['label'] != 'unknown'].copy()
        for c in INT_COLS:
            labeled[c] = labeled[c].astype('int64')
        if os.path.exists(LABELS_CSV):
            os.makedirs('backups', exist_ok=True)
            ts = datetime.now().strftime('%Y%m%d_%H%M%S')
            shutil.copy2(LABELS_CSV, os.path.join('backups', f'labeled_hits_{ts}.csv'))
        labeled[ORIGINAL_BRANCHES + ['label']].to_csv(LABELS_CSV, index=False)
        ns = int((labeled['label'] == 'signal').sum())
        nn = int((labeled['label'] == 'noise').sum())
        self.status.value = (f"<b style='color:green'>Saved {len(labeled)} labeled hits "
                             f"({ns} signal, {nn} noise) to '{LABELS_CSV}'.</b>")


Loading real data for interactive labeling...
Initial hits loaded: 27532828
Valid TPC hits remaining after cuts: 25671334
Resumed 1069 previously-labeled hits from 'labeled_hits.csv'
Successfully loaded 25671334 hits. Ready for labeling!


In [19]:
ui_xyz = LabelingUI("xyz")
display(ui_xyz.ui)

In [17]:
ui_unrolled = LabelingUI("layer_phi_z")
display(ui_unrolled.ui)

In [18]:
# Manual save (identical to the 💾 Save button in the UI).
# Writes a timestamped backup of any existing file first, and casts key columns to int.
labeled_hits = df_real[df_real['label'] != 'unknown'].copy()
for c in INT_COLS:
    labeled_hits[c] = labeled_hits[c].astype('int64')

if os.path.exists(LABELS_CSV):
    os.makedirs('backups', exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    shutil.copy2(LABELS_CSV, os.path.join('backups', f'labeled_hits_{ts}.csv'))

labeled_hits[ORIGINAL_BRANCHES + ['label']].to_csv(LABELS_CSV, index=False)
print(f"Saved {len(labeled_hits)} labeled hits to '{LABELS_CSV}' "
      f"({(labeled_hits['label'] == 'signal').sum()} signal, "
      f"{(labeled_hits['label'] == 'noise').sum()} noise)")


Saved 1128 labeled hits to 'labeled_hits.csv' (366 signal, 762 noise)
